# TVAE Interpretability Framework - Demo
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook provides a complete interactive workflow for training a Tabular Variational Autoencoder (TVAE) and extracting gradient-based feature importance and feature interaction metrics.

In [ ]:
# Clone repository and install requirements
!git clone https://github.com/jhw0010/jinhongwu.github.io.git
%cd jinhongwu.github.io
!pip install -r requirements.txt

In [ ]:
import pandas as pd
import numpy as np
from src.data import DataPipeline
from src.model import TVAE
from src.interpretability import TVAEInterpreter
from src.loss import compute_gradients
import tensorflow as tf

# Generate synthetic dummy tabular dataset
np.random.seed(42)
dummy_data = pd.DataFrame(np.random.randn(1000, 19), columns=[f'feature_{i}' for i in range(19)])
dummy_data['y'] = np.random.choice([0, 1], size=1000)
dummy_data.to_csv('dummy_financial.csv', index=False)

In [ ]:
# Pipeline Execution
pipeline = DataPipeline(target_column='y')
processed_df, _ = pipeline.fit_transform(dummy_data)
dataset = pipeline.create_dataset(processed_df, batch_size=128)

model = TVAE(input_dim=19, latent_dim=14)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

for epoch in range(1, 21):
    for train_x in dataset:
        grads, loss = compute_gradients(model, train_x)
        optimizer.apply_gradients(zip(grads, model.trainable_variables))
    if epoch % 5 == 0:
        print(f'Epoch {epoch} - Loss: {loss.numpy():.4f}')

In [ ]:
# Interpretability Extraction
interpreter = TVAEInterpreter(model)
mu_grads, logvars_grads = interpreter.compute_first_order_gradients(dataset)
importance = interpreter.compute_global_importance(mu_grads, logvars_grads)
interpreter.plot_importance(importance, feature_names=list(processed_df.columns))